In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import Dataset, DataLoader
import random

In [ ]:
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print(device)

In [ ]:
# rubber
def bezier_curve(P0, P1, P2, P3, t):
    return ((1 - t)**3) * P0 + 3 * (1 - t)**2 * t * P1 + 3 * (1 - t) * (t**2) * P2 + (t**3) * P3

def rubber(img, rubber_thickness=1.2):
    img = img.clone()
    _, H, W = img.shape

    # random control points within the image
    P0 = np.array([np.random.uniform(0, H), np.random.uniform(0, W)])
    P1 = np.array([np.random.uniform(0, H), np.random.uniform(0, W)])
    P2 = np.array([np.random.uniform(0, H), np.random.uniform(0, W)])
    P3 = np.array([np.random.uniform(0, H), np.random.uniform(0, W)])

    # sample the spline
    t = np.linspace(0, 1, 300)
    curve = np.array([bezier_curve(P0, P1, P2, P3, ti) for ti in t])

    # compute minimum distance from each pixel to the curve
    curve_tensor = torch.tensor(curve, dtype=torch.float32)  # shape [300, 2]
    Y, X = torch.meshgrid(
        torch.arange(H, dtype=torch.float32),
        torch.arange(W, dtype=torch.float32),
        indexing="ij"
    )
    distances = torch.cdist(torch.stack([Y.flatten(), X.flatten()], dim=1), curve_tensor).min(dim=1).values.reshape(H, W)

    # apply erasing
    mask = distances < rubber_thickness
    img[0][mask] = 0.0

    return img

In [ ]:
class UnlabeledTensorDataset(Dataset):
    def __init__(self, tensor):
        self.tensor = tensor

    def __len__(self):
        return len(self.tensor)

    def __getitem__(self, idx):
        return self.tensor[idx], self.tensor[idx]

In [ ]:
# download images
!wget http://agentspace.org/download/2000_rubbered_digits.pt

In [ ]:
loaded_images = torch.load("2000_rubbered_digits.pt")
new_dataset = UnlabeledTensorDataset(loaded_images)

print(len(new_dataset))      # 2000
print(new_dataset[0][0].shape)  # torch.Size([1, 28, 28])

In [ ]:
# show first samples
plt.figure(figsize=(20, 2))
for i in range(10):
    img = (new_dataset[i][0].squeeze(0).detach().numpy()*255).astype(np.uint8)
    plt.subplot(1, 10, i+1)
    plt.imshow(img, cmap='gray')
    plt.axis('off')
plt.show()